# no5 · Blooper (non-speech) detection

Input: a podcast/newscast video. Output: one clip per span where the speaker(s)
are **not** talking (silence, pauses, music, cutaways, dropped mics).

Pipeline: `extract_audio` → Silero VAD `speech_spans` → invert to
`nonspeech_spans` → OpenCV `lip_motion_score` (confirm mouth static) →
`cut_clips`.

**Prereqs:** `brew install ffmpeg`, then `pip install -r requirements.txt`.
Drop a short clip in `input/`.

In [ ]:
import importlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import blooper
importlib.reload(blooper)  # re-pick edits to blooper.py without restarting

# Point at the first video in input/
vids = sorted(p for p in Path('input').glob('*') if p.suffix.lower() in {'.mp4', '.mov', '.mkv', '.m4v'})
assert vids, 'Put a podcast/newscast video in input/'
VIDEO = str(vids[0])
DUR = blooper.video_duration(VIDEO)
print(f'{VIDEO}  ({DUR:.1f}s)')

## 1–2. Extract audio + run Silero VAD

In [ ]:
wav = blooper.extract_audio(VIDEO)
speech = blooper.speech_spans(wav)
Path(wav).unlink(missing_ok=True)
print(f'{len(speech)} speech segment(s)')
speech[:5]

## 3. Invert → candidate non-speech spans

Tune `min_dur` / `merge_gap` here for podcast vs newscast footage.

In [ ]:
MIN_DUR, PAD, MERGE_GAP = 0.8, 0.1, 0.3
gaps = blooper.nonspeech_spans(speech, DUR, min_dur=MIN_DUR, pad=PAD, merge_gap=MERGE_GAP)
print(f'{len(gaps)} candidate blooper span(s)')

fig, ax = plt.subplots(figsize=(12, 1.6))
for s, e in speech:
    ax.axvspan(s, e, color='tab:green', alpha=0.6)
for s, e in gaps:
    ax.axvspan(s, e, color='tab:red', alpha=0.6)
ax.set_xlim(0, DUR); ax.set_yticks([]); ax.set_xlabel('seconds')
ax.set_title('green = speech   red = non-speech (blooper)')
plt.tight_layout(); plt.show()

## 4. Visual lip-motion check

For each candidate span, OpenCV detects the face and measures how much the
mouth region changes frame-to-frame (mean abs diff, 0–1).
Low → `silent`; high → `lips_moving` (speaker mouthing while audio silent);
no face → `no_face` (audio-only verdict). Tune `LIP_THRESHOLD` per footage.

In [ ]:
LIP_THRESHOLD = 0.04   # mean mouth-region frame-diff; raise if too many 'lips_moving'
rows = []
for s, e in gaps:
    lm = blooper.lip_motion_score(VIDEO, s, e)
    rows.append(blooper.Blooper(s, e, round(e - s, 3), lm, blooper._label(lm, LIP_THRESHOLD)))

df = pd.DataFrame([b.__dict__ for b in rows])
df

## 5. Cut individual clips + write index

In [ ]:
index = blooper.cut_clips(
    VIDEO, rows, outdir='out',
    params={'min_dur': MIN_DUR, 'pad': PAD, 'merge_gap': MERGE_GAP, 'lip_threshold': LIP_THRESHOLD},
)
print(f"{index['count']} clip(s) written to out/clips/")
for f in sorted(Path('out/clips').glob('*.mp4')):
    print(' ', f.name)

## One-shot equivalent

Everything above is also `blooper.detect(VIDEO)` + `blooper.cut_clips(...)`, or
from the shell: `python blooper.py input/yourvideo.mp4`.